# Transformar la Data de Constructores

1. Leer la tabla de bronze `constructors`
1. Mantener solo las colunas requeridas para analítica (Dropear columna `url`)
1. Estandarizar los nombres de la columnas a  snake_case y traducir a español (`constructorId` → `id_contructor`)
1. Renombrar columnas a su nombre más relevante y traducirlas las español (`name` → `nombre_constructor` , `nationality` → `nacionalidad`)
1. Remover registros duplicados
1. Transformar los valores dela columna `nationalidad` a Title Case y hacer join con el dataframe de nacionalidad_region para traer la columna region, traducir los valores de las columnas nacionalidad y region al español
1. Escribir la data transformada a la tabla silver `constructores` 



#### Entity Relationship Diagram - Formula1 Bronze Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/04.Ayuda_Silver

In [0]:
tabla_bronze = f"{catalogo}.{esquema_bronze}.constructors"
tabla_silver = f"{catalogo}.{esquema_silver}.constructores"
ref_nationality_region_df = spark.table(f"{catalogo}.{esquema_silver}.ref_nacionalidad_region")

In [0]:
 #%run "../02-Silver/91.Build_Nationality_Region_Reference"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `constructors` table

In [0]:
df_constructors = spark.table(tabla_bronze)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
df_constructors_dropped = df_constructors.drop("url")

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`constructorId` → `constructor_id`)
- Rename columns to make them more meaningful (`name` → `constructor_name`)

In [0]:
df_constructors_renombre = (
    df_constructors_dropped
        .withColumnsRenamed({
            "constructorId": "id_constructor",
            "name": "nombre_constrcutor",
            "nationality": "nacionalidad"

        })
)

In [0]:
display(df_constructors_renombre)

#### Step 5 - Remove duplicate records

In [0]:
df_constructores_distinct = df_constructors_renombre.dropDuplicates(["id_constructor"])

In [0]:
display(df_constructores_distinct)

#### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
df_constructores_join = (
    df_constructores_distinct 
        .join(
            ref_nationality_region_df,  # 1. El DataFrame con el que te vas a unir va aquí
            F.initcap(df_constructores_distinct.nacionalidad) == ref_nationality_region_df.nationality, # 2. Condición corregida
            "left"
        ).select(
            df_constructores_distinct.id_constructor,
            df_constructores_distinct.nombre_constrcutor, # Nota: Revisa si en tu origen está escrito 'nombre_constrcutor' o 'nombre_constructor'
            df_constructores_distinct.nacionalidad,
            ref_nationality_region_df.region,
            df_constructores_distinct.ingesta_timestamp,
            df_constructores_distinct.archivo_fuente
        )
)

display(df_constructores_join)

In [0]:
from pyspark.sql.functions import when, col
df_constructores_final= (

   df_constructores_join      
         .withColumn("nacionalidad", traducir_f1_udf( F.initcap( F.col("nacionalidad") ) ))
         .withColumn( "region",
             when(col("region") =="Europe",  "Europa")
             . when(col("region") =="North America",  "Norteamérica")
             . when(col("region") =="South America",  "Suramérica")
             . when(col("region") =="Africa",  "Africa")
             .when(col("region") =="Asia",  "Asia")
             .when(col("region") =="Oceania",  "Oceania")
             .otherwise("Desconocido")
))

#### Step 7 - Write the transformed data to silver `constructors` table

In [0]:
display(df_constructores_final)

In [0]:
(
    df_constructores_final
        .write
        .format("delta")
        .mode("overwrite")
     
        .saveAsTable(tabla_silver)
)

In [0]:
display(spark.table(tabla_silver))